### load models and basic infrence

In [68]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline
import torch

model_name = "Qwen/Qwen2.5-0.5B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Save model and tokenizer locally
model_save_path = "./model_cache/qwen2.5-0.5b"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)


('./model_cache/qwen2.5-0.5b\\tokenizer_config.json',
 './model_cache/qwen2.5-0.5b\\special_tokens_map.json',
 './model_cache/qwen2.5-0.5b\\vocab.json',
 './model_cache/qwen2.5-0.5b\\merges.txt',
 './model_cache/qwen2.5-0.5b\\added_tokens.json',
 './model_cache/qwen2.5-0.5b\\tokenizer.json')

### use the model to generate text


In [11]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

text = pipe("hello, how are you?", max_new_tokens=25)
# text = pipe("hello, who is trump?", max_new_tokens=25)
print(type(text))
print(text[0]['generated_text'])



Device set to use cpu


<class 'list'>
hello, how are you? I'm doing a project for school and I need help with this problem. I have a function that takes in a string and


### tokenize the text


In [33]:
input_prompt = ["how are you?",
                "who is trump?",
                "what is the weather in beijing?"
                ]

input_ids = tokenizer.encode(input_prompt[0], return_tensors="pt", padding=True).to(model.device)
print(input_ids.shape)

input_ids_batch = tokenizer.batch_encode_plus(input_prompt, return_tensors="pt", padding=True).to(model.device)
print(input_ids_batch.keys())
print(input_ids_batch['input_ids'].shape)
print(input_ids_batch['input_ids'])
print(input_ids_batch['input_ids'][2])
print(tokenizer.batch_decode(input_ids_batch['input_ids'][2]))
print(input_ids_batch['attention_mask'][2])



torch.Size([1, 4])
dict_keys(['input_ids', 'attention_mask'])
torch.Size([3, 8])
tensor([[  5158,    525,    498,     30, 151643, 151643, 151643, 151643],
        [ 14623,    374,  38055,     30, 151643, 151643, 151643, 151643],
        [ 12555,    374,    279,   9104,    304,    387,  23649,     30]])
tensor([12555,   374,   279,  9104,   304,   387, 23649,    30])
['what', ' is', ' the', ' weather', ' in', ' be', 'ijing', '?']
tensor([1, 1, 1, 1, 1, 1, 1, 1])


In [73]:
input_prompt = "hello, how are"
input_ids = tokenizer.encode(input_prompt, return_tensors="pt", padding=True).to(model.device)
print(f"input_ids: {input_ids}")
print(f"input_ids.shape: {input_ids.shape}")
output_ids = model.generate(input_ids, 
                            max_new_tokens=25
                            )
print(f"output_ids.shape: {output_ids.shape}")
print(f"output_ids: {output_ids}")
print(f"output_text: {tokenizer.batch_decode(output_ids)}")


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


input_ids: tensor([[14990,    11,  1246,   525]])
input_ids.shape: torch.Size([1, 4])
output_ids.shape: torch.Size([1, 29])
output_ids: tensor([[14990,    11,  1246,   525,   498,    30,   358,  2776,  3730,   264,
          2390,   369,  2906,   323,   358,  1184,  1492,   448,   419,  3491,
            13,   358,   614,   264,   729,   429,  4990,   304,   264]])
output_text: ["hello, how are you? I'm doing a project for school and I need help with this problem. I have a function that takes in a"]


In [99]:
with torch.no_grad():  # 关闭梯度计算，进入推理模式
    # 将输入传入模型，获取输出
    outputs = model(input_ids)
    # 获取输出的logits（原始预测分数）
    logits = outputs.logits
    print(f"logits.shape: {logits.shape}")

    # 获取最后一个token的预测结果
    # [:, -1, :] 表示：所有批次，最后一个位置，所有词汇表分数
    last_token_logits = logits[:, -1, :]
    print(f"last_token_logits.shape: {last_token_logits.shape}")
    
    # 使用softmax将原始分数转换为概率分布
    # dim=-1 表示在最后一个维度上进行softmax运算
    probabilities = torch.nn.functional.softmax(last_token_logits, dim=-1)
    
    # 获取概率最高的前5个预测
    # top_probs: 概率值
    # top_indices: 对应的词元索引
    top_probs, top_indices = torch.topk(probabilities, k=5)
    print(top_probs[:5])
    print(top_indices[:5])
    
    # 遍历每个预测结果并打印
    for i, (probs, indices) in enumerate(zip(top_probs, top_indices)):
        print(f"\nTop 5 most likely next tokens:")
        # 对每个预测的token
        for prob, idx in zip(probs, indices):
            # 将token的索引转换为实际文本
            token = tokenizer.decode([idx])
            # 打印token及其概率（保留4位小数）
            print(f"tokenid: {idx}, token: '{token}' {tokenizer.convert_ids_to_tokens([idx])}, Probability: {prob:.4f}")

logits.shape: torch.Size([1, 4, 151936])
last_token_logits.shape: torch.Size([1, 151936])
tensor([[0.9781, 0.0109, 0.0015, 0.0012, 0.0011]])
tensor([[ 498,  575,  582,  379, 2513]])

Top 5 most likely next tokens:
tokenid: 498, token: ' you' ['Ġyou'], Probability: 0.9781
tokenid: 575, token: ' u' ['Ġu'], Probability: 0.0109
tokenid: 582, token: ' we' ['Ġwe'], Probability: 0.0015
tokenid: 379, token: ' y' ['Ġy'], Probability: 0.0012
tokenid: 2513, token: ' things' ['Ġthings'], Probability: 0.0011


In [96]:
# 查看原始形式
print(tokenizer.convert_ids_to_tokens([498]))  # 会显示 'Ġyou'

# 查看解码后的形式
print(tokenizer.decode([498]))  # 会显示 ' you'

['Ġyou']
 you


### loss

In [117]:
tokenizer.special_tokens_map

{'eos_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

In [119]:
sentence = "<|im_start|>hello, my name is Julie<|im_end|>"
tokennized_ids = tokenizer.encode(sentence, return_tensors="pt", padding=True).to(model.device)
print(f"tokennized_ids: {tokennized_ids}")
print(f"tokenizer.batch_decode(tokennized_ids): {tokenizer.batch_decode(tokennized_ids)}")
for i in tokennized_ids:
    print(f"tokenizer.convert_ids_to_tokens(tokennized_ids): {tokenizer.convert_ids_to_tokens(i)}")


tokennized_ids: tensor([[151644,  14990,     11,    847,    829,    374,  41187, 151645]])
tokenizer.batch_decode(tokennized_ids): ['<|im_start|>hello, my name is Julie<|im_end|>']
tokenizer.convert_ids_to_tokens(tokennized_ids): ['<|im_start|>', 'hello', ',', 'Ġmy', 'Ġname', 'Ġis', 'ĠJulie', '<|im_end|>']


In [120]:
input_ids = tokennized_ids[:,:-1] # 去掉最后一个token
target_ids = tokennized_ids[:,1:]
print(f"input_ids: {input_ids}")
print(f"input_ids.shape: {input_ids.shape}")
print(f"target_ids: {target_ids}")
print(f"target_ids.shape: {target_ids.shape}")



input_ids: tensor([[151644,  14990,     11,    847,    829,    374,  41187]])
input_ids.shape: torch.Size([1, 7])
target_ids: tensor([[ 14990,     11,    847,    829,    374,  41187, 151645]])
target_ids.shape: torch.Size([1, 7])


In [123]:
promt = [
    {"role": "system", 
     "content": "You are a helpful assistant."
     },
    {"role": "user", 
     "content": "hello, who are you?"
    },
    {"role": "assistant", 
     "content": "my name is Julie"
    }
]

chat_template = tokenizer.apply_chat_template(promt,
                              tokenize=False,
                              add_generation_prompt=False,
                              padding=True,
                              return_tensors="pt",
                              )
print(f"chat_template: {chat_template}")

tokenized_ids = tokenizer.apply_chat_template(promt,
                              tokenize=True,
                              add_generation_prompt=False,
                              padding=True,
                              return_tensors="pt",
                              )
print(f"tokenized_ids: {tokenized_ids}")



chat_template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, who are you?<|im_end|>
<|im_start|>assistant
my name is Julie<|im_end|>

tokenized_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,    879,    525,
            498,     30, 151645,    198, 151644,  77091,    198,   2408,    829,
            374,  41187, 151645,    198]])


In [126]:
input_ids = tokenized_ids[:,:-1] # 去掉最后一个token
target_ids = tokenized_ids[:,1:]
print(f"input_ids: {input_ids}")
print(f"tokenizer.batch_decode(input_ids): {tokenizer.batch_decode(input_ids)}")
print(f"input_ids.shape: {input_ids.shape}")
print(f"\ntarget_ids: {target_ids}")
print(f"tokenizer.batch_decode(target_ids): {tokenizer.batch_decode(target_ids)}")
print(f"target_ids.shape: {target_ids.shape}")



input_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,    879,    525,
            498,     30, 151645,    198, 151644,  77091,    198,   2408,    829,
            374,  41187, 151645]])
tokenizer.batch_decode(input_ids): ['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\nmy name is Julie<|im_end|>']
input_ids.shape: torch.Size([1, 30])

target_ids: tensor([[  8948,    198,   2610,    525,    264,  10950,  17847,     13, 151645,
            198, 151644,    872,    198,  14990,     11,    879,    525,    498,
             30, 151645,    198, 151644,  77091,    198,   2408,    829,    374,
          41187, 151645,    198]])
tokenizer.batch_decode(target_ids): ['system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nhello, who are you?<|im_end|>\n<|im_start|>assistant\nmy name is Julie<|im_end|>\

### loss

In [130]:
question = "hello, what is your name?"
answer = "Julie"

prompt = [  
    {"role": "user", 
     "content": "hello, what is your name?"
    },
    {"role": "assistant", 
     "content": "name:"
    }
]

chat_template = tokenizer.apply_chat_template(prompt,
                              tokenize=False,
                              # add_generation_prompt=True,
                              continue_final_message=True,
                              padding=True,
                              return_tensors="pt",
                              )
print(f"chat_template: {chat_template}")


chat_template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, what is your name?<|im_end|>
<|im_start|>assistant
name:


In [131]:
full_response_text = chat_template + " " + answer +  "<|im_end|>"
print(f"full_response_text: {full_response_text}")


full_response_text: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, what is your name?<|im_end|>
<|im_start|>assistant
name: Julie<|im_end|>


In [132]:
tokenized_ids = tokenizer.encode(full_response_text, return_tensors="pt", padding=True).to(model.device)
print(f"tokenized_ids: {tokenized_ids}")
print(f"tokenizer.batch_decode(tokenized_ids): {tokenizer.batch_decode(tokenized_ids)}")



tokenized_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,   1128,    374,
            697,    829,     30, 151645,    198, 151644,  77091,    198,    606,
             25,  41187, 151645]])
tokenizer.batch_decode(tokenized_ids): ['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nhello, what is your name?<|im_end|>\n<|im_start|>assistant\nname: Julie<|im_end|>']


In [133]:
input_ids = tokenized_ids[:,:-1] # 去掉最后一个token
target_ids = tokenized_ids[:,1:]
print(f"input_ids: {input_ids}")
print(f"input_ids.shape: {input_ids.shape}")
print(f"target_ids: {target_ids}")
print(f"target_ids.shape: {target_ids.shape}")



input_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,   1128,    374,
            697,    829,     30, 151645,    198, 151644,  77091,    198,    606,
             25,  41187]])
input_ids.shape: torch.Size([1, 29])
target_ids: tensor([[  8948,    198,   2610,    525,    264,  10950,  17847,     13, 151645,
            198, 151644,    872,    198,  14990,     11,   1128,    374,    697,
            829,     30, 151645,    198, 151644,  77091,    198,    606,     25,
          41187, 151645]])
target_ids.shape: torch.Size([1, 29])


In [154]:
new_answer = " " + answer
lable_tokenized = tokenizer.encode(new_answer, return_tensors="pt").to(model.device)
print(f"lable_tokenized shape: {lable_tokenized.shape}")
print(f"lable_tokenized: {lable_tokenized}")
print(f"tokenizer.batch_decode(lable_tokenized): {tokenizer.batch_decode(lable_tokenized)}")



lable_tokenized shape: torch.Size([1, 1])
lable_tokenized: tensor([[41187]])
tokenizer.batch_decode(lable_tokenized): [' Julie']


In [156]:
tokenizer.convert_ids_to_tokens(lable_tokenized)

['ĠJulie']

In [161]:
print(f"target_ids.shape: {target_ids.shape}")
print(f"tokenizer.eos_token: {tokenizer.eos_token}")
lable_tokenized = tokenizer.encode(new_answer + tokenizer.eos_token,
                                   add_special_tokens=False, 
                                   return_tensors="pt",
                                   padding="max_length",
                                   padding_side="left",
                                   max_length=tokenized_ids.shape[1],
                                   ).to(model.device)
print(f"lable_tokenized: {lable_tokenized}")
print(f"tokenizer.batch_decode(lable_tokenized): {tokenizer.batch_decode(lable_tokenized)}")



target_ids.shape: torch.Size([1, 29])
tokenizer.eos_token: <|endoftext|>
lable_tokenized: tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643,  41187, 151643]])
tokenizer.batch_decode(lable_tokenized): ['<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|> Julie<|endoftext|>']


In [169]:
lable_tokenized_fixed = torch.where(lable_tokenized == tokenizer.eos_token_id, 
                                    lable_tokenized, 
                                    torch.tensor(-100))
print(f"lable_tokenized_fixed: {lable_tokenized_fixed}")


lable_tokenized_fixed: tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643,   -100, 151643]])


In [ ]:
def gennerate_input_output_pair(prompt, target_responses):
    chat_templates = tokenizer.apply_chat_template(prompt,
                              tokenize=False,
                              continue_final_message=True,
                              padding=True,
                              return_tensors="pt",
                              )
    full_response_text = [
        chat_template + " " + target_response +  "<|im_end|>"
        for chat_template, target_response in zip(chat_templates, target_responses)    
    ]
    tokenized_ids = tokenizer.encode(full_response_text, return_tensors="pt", padding=True).to(model.device)
    lable_tokenized = tokenizer.encode(new_answer + tokenizer.eos_token,
                                   add_special_tokens=False, 
                                   return_tensors="pt",
                                   padding="max_length",
                                   padding_side="left",
                                   max_length=tokenized_ids.shape[1],
                                   ).to(model.device)
    input_ids = tokenized_ids[:,:-1] # 去掉最后一个token
    target_ids = tokenized_ids[:,1:]
    return input_ids, target_ids


In [ ]:
def caculate_loss(input_ids, target_ids):
    outputs = model(input_ids, labels=target_ids)
    loss = outputs.loss
    return loss


### chat template

In [66]:
import json
prompt_template = [
    {"role": "system", 
     "content": "You are a helpful assistant."
     },
    {"role": "user", 
     "content": "hello, who are you?"
    },
    {"role": "assistant", 
     "content": "I am Julie"
    }
]

tokennized_prompt = tokenizer.apply_chat_template(prompt_template,
                              tokenize=False,
                              add_generation_prompt=True,
                              padding=True,
                              return_tensors="pt",
                              )
print(f"tokennized_prompt: {tokennized_prompt}")

tokenized_ids = tokenizer.apply_chat_template(prompt_template,
          tokenize=True,
          padding=True,
          add_generation_prompt=True,
          return_tensors="pt",
          )
print(f"\ntokenized_ids: {tokenized_ids}")
decoded_prompt = tokenizer.batch_decode(tokenized_ids)
print(f"\ndecoded_prompt: {decoded_prompt[0]}")



tokennized_prompt: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, how are you?<|im_end|>
<|im_start|>assistant
I AM<|im_end|>
<|im_start|>assistant


tokenized_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,   1246,    525,
            498,     30, 151645,    198, 151644,  77091,    198,     40,   6769,
         151645,    198, 151644,  77091,    198]])

decoded_prompt: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, how are you?<|im_end|>
<|im_start|>assistant
I AM<|im_end|>
<|im_start|>assistant



In [67]:
OUTPUT_IDS = model.generate(tokenized_ids, max_new_tokens=25)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(tokenized_ids, OUTPUT_IDS)
]
print(f"\nOUTPUT_IDS: {generated_ids}")
OUTPUT_TEXT = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print(f"\nOUTPUT_TEXT: {OUTPUT_TEXT}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



OUTPUT_IDS: [tensor([    40,   1079, 112169,   8948,    198,  34978,    198,  34978,    198,
         34978,    198,  34978,    198,  34978,    198,  34978,    198,  34978,
           198,  34978,    198,  34978,    198,  34978,    198])]

OUTPUT_TEXT: ['I am很高兴system\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n']
